# 06 — Molecular Properties: Dipoles, MOs, Fukui Functions & ESP

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ppt-2/Ch121a-DFT/blob/main/notebooks/06_molecular_properties.ipynb)

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:
- Compute and interpret molecular dipole moments
- Analyze HOMO/LUMO energies and the HOMO-LUMO gap
- Calculate chemical hardness η and chemical potential μ
- Compute Fukui functions for reactivity prediction
- Understand electrostatic potential (ESP) maps
- Understand the concept of NMR chemical shielding (brief overview)
- Connect DFT-computed properties to experimental observables

## 1. Theory: Molecular Properties from DFT

### 1.1 Dipole Moment

The molecular dipole moment measures the asymmetry of the charge distribution:

$$\boldsymbol{\mu} = -\int \mathbf{r}\, n(\mathbf{r})\, d\mathbf{r} + \sum_I Z_I \mathbf{R}_I$$

The magnitude in Debye: $\mu = |\boldsymbol{\mu}|$, where 1 D = 0.3934 a.u.

### 1.2 HOMO-LUMO Gap and Chemical Reactivity

From Koopmans' theorem extended to DFT (Janak's theorem):

$$IE \approx -\epsilon_{HOMO} \quad \text{(ionization energy)}$$
$$EA \approx -\epsilon_{LUMO} \quad \text{(electron affinity, approximate)}$$

**Conceptual DFT** (Parr and Yang) defines:
- **Chemical potential**: $\mu_{chem} = \frac{\epsilon_{HOMO} + \epsilon_{LUMO}}{2} \approx -\frac{IE + EA}{2}$
- **Chemical hardness**: $\eta = \frac{\epsilon_{LUMO} - \epsilon_{HOMO}}{2} \approx \frac{IE - EA}{2}$
- **Electrophilicity index**: $\omega = \frac{\mu_{chem}^2}{2\eta}$

Soft molecules (small η) are more reactive; hard molecules are less reactive.

### 1.3 Fukui Functions

The **Fukui function** describes site-reactivity for electron addition/removal:

$$f^+(\mathbf{r}) = n_{N+1}(\mathbf{r}) - n_N(\mathbf{r}) \quad \text{(nucleophilic attack)}$$
$$f^-(\mathbf{r}) = n_N(\mathbf{r}) - n_{N-1}(\mathbf{r}) \quad \text{(electrophilic attack)}$$
$$f^0(\mathbf{r}) = \frac{f^+(\mathbf{r}) + f^-(\mathbf{r})}{2} \quad \text{(radical attack)}$$

Using **condensed Fukui functions** from Mulliken charges $q_k$:
$$f_k^+ = q_k(N+1) - q_k(N)$$
$$f_k^- = q_k(N) - q_k(N-1)$$

### 1.4 Electrostatic Potential (ESP)

The ESP measures the interaction energy between the molecular charge distribution
and a unit positive test charge at position $\mathbf{r}$:

$$V(\mathbf{r}) = \sum_I \frac{Z_I}{|\mathbf{r}-\mathbf{R}_I|} - \int \frac{n(\mathbf{r}')}{|\mathbf{r}-\mathbf{r}'|} d\mathbf{r}'$$

- **Negative ESP** (red): electron-rich regions → sites for electrophilic attack
- **Positive ESP** (blue): electron-poor regions → sites for nucleophilic attack
- ESP mapped onto the molecular isodensity surface gives an **ESP map**

In [ ]:
%%time
# =============================================================================
# Ch121a: Quantum Chemistry & DFT — Notebook 06: Molecular Properties
# License: GPL-3.0 (https://www.gnu.org/licenses/gpl-3.0.en.html)
# =============================================================================
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyscf import gto, dft, scf

# ------------------------------------------------------------------
# Dipole Moments: Computed vs Experimental
# ------------------------------------------------------------------
# Compute B3LYP/def2-SVP dipole moments for small polar molecules

molecules_dip = {
    'H₂O':  ('O 0 0 0.117; H 0 0.757 -0.469; H 0 -0.757 -0.469', 1.85),
    'HF':   ('H 0 0 0; F 0 0 0.917',                               1.82),
    'CO':   ('C 0 0 0; O 0 0 1.128',                               0.12),
    'NH₃':  ('N 0 0 0.116; H 0 0.939 -0.269; H 0.814 -0.469 -0.269; H -0.814 -0.469 -0.269', 1.47),
    'CO₂':  ('C 0 0 0; O 0 0 1.162; O 0 0 -1.162',                0.00),
    'CH₄':  ('C 0 0 0; H 0.629 0.629 0.629; H -0.629 -0.629 0.629; H 0.629 -0.629 -0.629; H -0.629 0.629 -0.629', 0.00),
}

dip_results = []
for name, (atom_str, exp_dip) in molecules_dip.items():
    mol = gto.Mole()
    mol.atom = atom_str
    mol.basis = 'def2-SVP'
    mol.verbose = 0
    mol.build()
    
    mf = dft.RKS(mol)
    mf.xc = 'B3LYP'
    mf.verbose = 0
    mf.kernel()
    
    dm = mf.make_rdm1()
    dip_vec = mf.dip_moment(mol, dm, verbose=0)
    dip_mag = np.linalg.norm(dip_vec)
    
    dip_results.append({
        'Molecule': name,
        'Computed (D)': round(dip_mag, 3),
        'Experimental (D)': exp_dip,
        'Error (D)': round(dip_mag - exp_dip, 3),
    })
    print(f"  {name:5s}: μ = {dip_mag:.3f} D (exp: {exp_dip:.2f} D)")

df_dip = pd.DataFrame(dip_results)
print("\n")
print("Dipole Moments: B3LYP/def2-SVP vs Experiment")
print(df_dip.to_string(index=False))
mae_dip = np.mean(np.abs(df_dip['Error (D)']))
print(f"\nMean Absolute Error: {mae_dip:.3f} D")

# Bar chart: computed vs experimental
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(dip_results))
width = 0.35
names = [r['Molecule'] for r in dip_results]
comp = [r['Computed (D)'] for r in dip_results]
expt = [r['Experimental (D)'] for r in dip_results]

bars1 = ax.bar(x - width/2, comp, width, color='steelblue', label='Computed (B3LYP/def2-SVP)',
               edgecolor='white', linewidth=0.5)
bars2 = ax.bar(x + width/2, expt, width, color='coral', label='Experimental',
               edgecolor='white', linewidth=0.5, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=11)
ax.set_xlabel('Molecule', fontsize=12)
ax.set_ylabel('Dipole Moment (Debye)', fontsize=12)
ax.set_title('Dipole Moments: B3LYP/def2-SVP vs Experiment', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars1, comp):
    if val > 0.05:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
%%time
# ------------------------------------------------------------------
# HOMO/LUMO Analysis: Chemical Hardness and Chemical Potential
# ------------------------------------------------------------------

molecules_homo = {
    'Ethylene\n(C₂H₄)':  'C 0 0 0; C 0 0 1.34; H 0.924 0 -0.54; H -0.924 0 -0.54; H 0.924 0 1.88; H -0.924 0 1.88',
    'Butadiene\n(C₄H₆)':  'C 0 0 0; C 0 0 1.34; C 0 0.83 2.2; C 0 0.83 3.54; H 0.92 0 -0.54; H -0.92 0 -0.54; H 0 -0.93 1.88; H 0.92 1.76 1.65; H -0.92 1.76 1.65; H 0 -0.1 4.08',
    'Benzene\n(C₆H₆)':    'C 0 1.4 0; C 1.212 0.7 0; C 1.212 -0.7 0; C 0 -1.4 0; C -1.212 -0.7 0; C -1.212 0.7 0; H 0 2.48 0; H 2.147 1.24 0; H 2.147 -1.24 0; H 0 -2.48 0; H -2.147 -1.24 0; H -2.147 1.24 0',
    'Naphthalene\n(C₁₀H₈)': 'C 0.0 2.46 0; C 1.243 1.408 0; C 1.243 0.0 0; C 0.0 -0.705 0; C -1.243 0.0 0; C -1.243 1.408 0; C 1.243 -1.408 0; C 0.0 -2.113 0; C -1.243 -1.408 0; C 0.0 2.819 0; H 0.0 3.541 0; H 2.154 1.946 0; H 2.154 -1.946 0; H 0.0 -3.188 0; H -2.154 -1.946 0; H -2.154 1.946 0',
}

homo_results = []
for name, atom_str in molecules_homo.items():
    mol = gto.Mole()
    mol.atom = atom_str
    mol.basis = '6-31g*'
    mol.verbose = 0
    try:
        mol.build()
        mf = dft.RKS(mol)
        mf.xc = 'B3LYP'
        mf.verbose = 0
        mf.kernel()
        
        mo_e = mf.mo_energy
        mo_occ = mf.mo_occ
        homo_idx = np.where(mo_occ > 0)[0][-1]
        lumo_idx = homo_idx + 1
        
        homo_ev = mo_e[homo_idx] * 27.2114
        lumo_ev = mo_e[lumo_idx] * 27.2114
        gap = lumo_ev - homo_ev
        eta = gap / 2.0
        mu_chem = (homo_ev + lumo_ev) / 2.0
        omega = mu_chem**2 / (2 * eta)
        
        homo_results.append({
            'Molecule': name.replace('\n', ' '),
            'HOMO (eV)': round(homo_ev, 3),
            'LUMO (eV)': round(lumo_ev, 3),
            'Gap (eV)': round(gap, 3),
            'η (eV)': round(eta, 3),
            'μ_chem (eV)': round(mu_chem, 3),
            'ω (eV)': round(abs(omega), 3),
        })
        print(f"  {name.split(chr(10))[0]:12s}: HOMO={homo_ev:.2f} eV  LUMO={lumo_ev:.2f} eV  Gap={gap:.2f} eV  η={eta:.2f} eV")
    except Exception as e:
        print(f"  {name}: Error - {e}")

df_homo = pd.DataFrame(homo_results)
print("\n")
print("HOMO/LUMO Analysis: B3LYP/6-31G*")
print(df_homo.to_string(index=False))

# Plot: HOMO-LUMO gap comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

mols = [r['Molecule'] for r in homo_results]
homos = [r['HOMO (eV)'] for r in homo_results]
lumos = [r['LUMO (eV)'] for r in homo_results]
gaps = [r['Gap (eV)'] for r in homo_results]

colors_pi = ['steelblue', 'seagreen', 'coral', 'mediumpurple']

# Orbital energy diagram
x_pos = np.arange(len(homo_results))
for i, (mol_name, homo, lumo) in enumerate(zip(mols, homos, lumos)):
    col = colors_pi[i % len(colors_pi)]
    ax1.hlines(homo, i-0.3, i+0.3, colors=col, linewidth=5, label=f'HOMO' if i==0 else '')
    ax1.hlines(lumo, i-0.3, i+0.3, colors=col, linewidth=5, linestyle='--', label=f'LUMO' if i==0 else '')
    ax1.vlines(i, homo, lumo, colors=col, linewidth=1.5, alpha=0.4)

ax1.set_xticks(x_pos)
ax1.set_xticklabels([m.split(' ')[0] for m in mols], fontsize=9)
ax1.set_ylabel('Orbital Energy (eV)', fontsize=12)
ax1.set_title('HOMO/LUMO Energies\nB3LYP/6-31G*', fontsize=12)
ax1.grid(True, alpha=0.3, axis='y')
ax1.axhline(y=0, color='black', linestyle=':', alpha=0.4)
from matplotlib.lines import Line2D
legend_elements = [Line2D([0],[0],color='gray',linewidth=5,label='HOMO (solid)'),
                   Line2D([0],[0],color='gray',linewidth=5,linestyle='--',label='LUMO (dashed)')]
ax1.legend(handles=legend_elements, fontsize=9)

# Gap bar chart
bars = ax2.bar(range(len(homo_results)), gaps, color=colors_pi[:len(homo_results)],
               edgecolor='white', linewidth=0.5)
ax2.set_xticks(range(len(homo_results)))
ax2.set_xticklabels([m.split(' ')[0] for m in mols], fontsize=9)
ax2.set_xlabel('Molecule', fontsize=12)
ax2.set_ylabel('HOMO-LUMO Gap (eV)', fontsize=12)
ax2.set_title('HOMO-LUMO Gaps\n(smaller gap → more reactive)', fontsize=12)
ax2.grid(True, alpha=0.3, axis='y')
for bar, gap_val in zip(bars, gaps):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{gap_val:.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
%%time
# ------------------------------------------------------------------
# Fukui Functions: Site-Specific Reactivity
# ------------------------------------------------------------------
# Condensed Fukui functions from Mulliken population analysis
# for formaldehyde (H₂C=O)

from pyscf import gto, dft
import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
import matplotlib.pyplot as plt

def get_mulliken_charges(atom_str, n_electrons_adjust=0, basis='def2-SVP', xc='B3LYP'):
    """Return Mulliken charges for a molecule."""
    mol = gto.Mole()
    mol.atom = atom_str
    mol.basis = basis
    mol.charge = n_electrons_adjust   # +1 = cation, -1 = anion
    mol.spin = 0 if n_electrons_adjust == 0 else 1  # doublet for +1 or -1
    mol.verbose = 0
    mol.build()
    
    mf = dft.RKS(mol)
    mf.xc = xc
    mf.verbose = 0
    mf.kernel()
    
    # Mulliken population analysis
    dm = mf.make_rdm1()
    mulliken = mf.mulliken_pop(mol, dm, verbose=0)
    charges = mulliken[1]   # Mulliken charges (list per atom)
    return charges, mol

# Formaldehyde H₂C=O
formaldehyde = '''
C   0.000000   0.000000   0.000000
O   0.000000   0.000000   1.208000
H   0.935000   0.000000  -0.540000
H  -0.935000   0.000000  -0.540000
'''
atom_labels = ['C', 'O', 'H₁', 'H₂']

print("Computing Mulliken charges for formaldehyde (N, N-1, N+1)...")

# N electrons (neutral)
q_N, mol_N = get_mulliken_charges(formaldehyde, 0)
# N-1 electrons (cation, electrophilic attack)
q_Nm1, mol_Nm1 = get_mulliken_charges(formaldehyde, +1)
# N+1 electrons (anion, nucleophilic attack)
q_Np1, mol_Np1 = get_mulliken_charges(formaldehyde, -1)

q_N = np.array(q_N)
q_Nm1 = np.array(q_Nm1)
q_Np1 = np.array(q_Np1)

# Condensed Fukui functions
f_plus = q_Np1 - q_N    # f+ : nucleophilic attack (system gains electron)
f_minus = q_N - q_Nm1   # f- : electrophilic attack (system loses electron)
f_zero = (f_plus + f_minus) / 2.0  # f0 : radical attack

print(f"\n  {'Atom':5s}  {'q(N)':>8s}  {'q(N-1)':>8s}  {'q(N+1)':>8s}  {'f+':>8s}  {'f-':>8s}  {'f0':>8s}")
print(f"  {'-'*5}  {'-'*8}  {'-'*8}  {'-'*8}  {'-'*8}  {'-'*8}  {'-'*8}")
for i, (label, qn, qnm, qnp, fp, fm, fz) in enumerate(
        zip(atom_labels, q_N, q_Nm1, q_Np1, f_plus, f_minus, f_zero)):
    print(f"  {label:5s}  {qn:8.4f}  {qnm:8.4f}  {qnp:8.4f}  {fp:8.4f}  {fm:8.4f}  {fz:8.4f}")

df_fukui = pd.DataFrame({
    'Atom': atom_labels,
    'q(N)': np.round(q_N, 4),
    'f+ (nucl.)': np.round(f_plus, 4),
    'f- (electr.)': np.round(f_minus, 4),
    'f0 (radical)': np.round(f_zero, 4),
})
print("\nFukui Functions Summary (B3LYP/def2-SVP):")
print(df_fukui.to_string(index=False))

# Bar chart
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
colors_at = ['steelblue', 'coral', 'seagreen', 'mediumpurple']

for ax, (values, title, ylabel) in zip(axes, [
    (f_plus, 'f⁺ (Nucleophilic Attack)\nSite for electrophile?', 'Fukui f⁺'),
    (f_minus, 'f⁻ (Electrophilic Attack)\nSite for nucleophile?', 'Fukui f⁻'),
    (f_zero, 'f⁰ (Radical Attack)', 'Fukui f⁰'),
]):
    bars = ax.bar(atom_labels, values, color=colors_at, edgecolor='white', linewidth=0.5)
    ax.axhline(y=0, color='black', linewidth=0.8, linestyle='-')
    ax.set_xlabel('Atom', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title, fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2,
                val + (0.005 if val >= 0 else -0.015),
                f'{val:.3f}', ha='center', va='bottom' if val >= 0 else 'top',
                fontsize=9)

plt.suptitle('Condensed Fukui Functions for Formaldehyde (H₂C=O)\nB3LYP/def2-SVP',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  f+ > 0 at O: nucleophilic attack site (oxygen is electrophile-susceptible)")
print("  f- > 0 at C: electrophilic attack site (carbon is nucleophile-susceptible)")
print("  This is consistent with H₂C=O: O attacks electrophiles, C attacks nucleophiles")

In [ ]:
# ------------------------------------------------------------------
# Electrostatic Potential (ESP): Concept and Visualization
# ------------------------------------------------------------------
# While full ESP grid calculations require specialized tools (Multiwfn, ORCA),
# we can visualize ESP conceptually using py3Dmol with a built-in surface.

try:
    import py3Dmol
    
    # Show water with a surface visualization
    water_xyz = """3
Water molecule
O   0.000000   0.000000   0.117176
H   0.000000   0.757001  -0.468704
H   0.000000  -0.757001  -0.468704
"""
    
    view = py3Dmol.view(width=500, height=400)
    view.addModel(water_xyz, 'xyz')
    
    # Stick model + VDW surface
    view.setStyle({'stick': {'radius': 0.15}, 'sphere': {'scale': 0.25}})
    view.addSurface(py3Dmol.VDW, {'opacity': 0.6,
                                   'colorscheme': {'gradient': 'rwb',
                                                   'min': -0.05, 'max': 0.05}})
    view.setBackgroundColor('white')
    view.zoomTo()
    view.show()
    print("ESP map displayed: red = electron-rich (negative), blue = electron-poor (positive)")
    
except ImportError:
    print("py3Dmol not installed. In a Colab/Jupyter environment, install with:")
    print("  pip install py3Dmol")

# Explain ESP calculation workflow
print("\n" + "=" * 60)
print("ESP Calculation Workflow:")
print("=" * 60)
print()
print("1. Run DFT calculation to obtain density matrix")
print("2. Compute ESP on a grid of points surrounding the molecule:")
print("   V(r) = Σ_I Z_I/|r-R_I| - ∫ n(r')/|r-r'| dr'")
print()
print("3. Map ESP onto molecular surface (typically ρ = 0.001 a.u. isosurface)")
print()
print("ESP maps can be generated with:")
print("  ORCA:      '! ESP' with %eprnmr or multiwfn post-processing")
print("  Gaussian:  'pop=mk' for Merz-Kollman ESP charges")
print("  Multiwfn:  Free tool for analyzing ESP from ORCA/Gaussian output")
print("  UCSF Chimera / VMD: Visualization of ESP-mapped surfaces")
print()
print("ESP Applications:")
print("  • Identifying reactive sites (nucleophilic/electrophilic)")
print("  • Deriving partial charges for molecular mechanics (RESP charges)")
print("  • Understanding drug-receptor interactions")
print("  • Predicting regioselectivity in reactions")

## 2. NMR Chemical Shielding (Brief Overview)

### Nuclear Magnetic Resonance (NMR) from DFT

NMR chemical shifts arise from the electronic shielding of the nuclear magnetic moment
by the surrounding electrons. DFT can compute the **isotropic chemical shielding tensor**:

$$\sigma_{iso} = \frac{1}{3}(\sigma_{xx} + \sigma_{yy} + \sigma_{zz})$$

The chemical shift relative to a reference compound (TMS for ¹H/¹³C):
$$\delta = \sigma_{ref} - \sigma_{iso}$$

### ORCA NMR Input

```
! B3LYP def2-TZVP NMR

%eprnmr
  Nuclei = all C { shift }    # 13C shielding
  Nuclei = all H { shift }    # 1H shielding
  GIAO = true                  # Gauge-including atomic orbitals (recommended)
end

* xyz 0 1
  [molecule coordinates]
*
```

### Key considerations for NMR calculations:

| Aspect | Recommendation |
|--------|---------------|
| Functional | B3LYP, PBE0, or ωB97X-D |
| Basis set | **def2-TZVP** minimum; pcSseg-2 or pcS-2 specialized NMR basis |
| Gauge | Use **GIAO** (gauge-including atomic orbitals) always |
| Reference | Calculate σ(TMS) at same level; δ = σ(TMS) - σ(molecule) |
| Solvent | Include implicit solvation (CPCM) for solution-phase spectra |

### Typical accuracy (¹H shifts):
- B3LYP/def2-TZVP: ±0.2-0.5 ppm for most organic molecules
- ωB97X-D/def2-TZVP: ±0.15-0.3 ppm

**See Chapter 12** of the course notes for a full ORCA NMR tutorial with worked examples.

## 🔬 Research Connection

Molecular properties from DFT are used throughout chemistry and biochemistry:

- **Drug design**: Dipole moments and ESP maps guide pharmacophore design and predict
  solubility, membrane permeability (log P), and protein binding modes.
- **Organic synthesis**: Fukui functions predict regioselectivity of reactions
  (e.g., which carbon is attacked in Michael additions, Friedel-Crafts alkylation).
- **NMR structure determination**: ¹³C and ¹H chemical shifts computed at DFT level
  are routinely used to distinguish between structural isomers and confirm synthesis products.
- **Materials science**: HOMO/LUMO gaps predict optical band gaps for solar cell materials,
  OLED emitters, and photocatalysts.

**The HSAB principle** (Hard and Soft Acids and Bases, Pearson 1963) is quantified
by chemical hardness η computed from DFT orbital energies — hard species prefer hard
partners, soft species prefer soft partners in chemical reactions.

**Notable application**: DFT-computed Fukui functions correctly predicted the
C2-selective metalation of oxazoline heterocycles, later confirmed experimentally.

## 📋 Summary

| Property | Formula | Physical Meaning |
|----------|---------|------------------|
| Dipole moment | $\boldsymbol{\mu} = -\int \mathbf{r}\, n\, d\mathbf{r} + \sum Z_I \mathbf{R}_I$ | Charge asymmetry; polarity |
| HOMO energy | $-\epsilon_{HOMO} \approx IE$ | Electron donation ability |
| LUMO energy | $-\epsilon_{LUMO} \approx EA$ | Electron acceptance ability |
| HOMO-LUMO gap | $\epsilon_{LUMO} - \epsilon_{HOMO}$ | Kinetic stability; color |
| Chemical hardness | $\eta = (\epsilon_{LUMO} - \epsilon_{HOMO})/2$ | Resistance to electron flow |
| Fukui f⁺ | $n_{N+1} - n_N$ | Nucleophilic attack site |
| Fukui f⁻ | $n_N - n_{N-1}$ | Electrophilic attack site |
| ESP | $V(\mathbf{r})$: nuclear - electronic potential | Reactivity map; partial charges |
| NMR shielding | $\sigma_{iso}$ | Chemical shift prediction |

**Recommended methods:**
- Dipole moments: B3LYP/def2-SVP (±0.1-0.2 D)
- HOMO/LUMO: B3LYP or PBE0 / def2-SVP (qualitative trends)
- Fukui functions: B3LYP/def2-SVP with Hirshfeld or CM5 charges (better than Mulliken)
- NMR: B3LYP/def2-TZVP with GIAO (±0.3-0.5 ppm for ¹H)

## 📝 Exercises

1. **Dipole trend**: Compute dipole moments for the halogen fluorides: HF, ClF, BrF.
   As you go down the periodic table, does the dipole moment increase or decrease? 
   Explain in terms of electronegativity and bond length.

2. **HOMO-LUMO gap and color**: Compute B3LYP/6-31G* HOMO-LUMO gaps for:
   - Anthracene (C₁₄H₁₀)
   - Tetracene (C₁₈H₁₂)  
   - Pentacene (C₂₂H₁₄)
   Plot gap vs number of rings. These molecules are yellow, orange, and blue — 
   does the gap correlate with their absorption wavelengths?
   (Hint: E = hc/λ; 1 eV ≈ 1240 nm)

3. **Fukui for acrolein**: Compute Fukui functions for acrolein (CH₂=CH-CHO).
   Acrolein undergoes 1,2- vs 1,4-addition. Which atom(s) have the largest f⁻?
   Does this explain the preference for 1,4-addition?
   Atom string: `C 0 0 0; C 0 0 1.34; C 0 0.83 2.2; O 0 0.83 3.4; H ...`

4. **Hardness comparison**: Compute chemical hardness η for: 
   Na⁺, F⁻, BF₃, NF₃, H₂O, H₂S.
   According to HSAB theory, which pairs would preferentially interact?

5. **ESP charges**: Look up the ORCA `pop = chelpg` keyword for ESP-fitted charges.
   Write an ORCA input for computing CHELPG charges on acrolein at B3LYP/def2-SVP.
   How would these CHELPG charges differ from Mulliken charges?